# Vivir más sin ser más rico — análisis

Partimos del panel armado en `02_datos.ipynb`. La pregunta es simple: **¿cuántos años de vida "corresponden" a cada nivel de ingreso, y qué países se apartan de esa regla?**

In [1]:
import numpy as np
import pandas as pd
import altair as alt
import duckdb

panel = pd.read_csv("../datos/panel.csv")
ANIO = 2024
corte = panel[(panel["anio"] == ANIO)].dropna(subset=["vida", "pib"]).copy()
print(len(corte), "países con esperanza de vida e ingreso en", ANIO)


195 países con esperanza de vida e ingreso en 2024


## 1. El patrón

En escala lineal la relación es una curva que se aplana: pasar de 2.000 a 10.000 dólares cambia mucho más la esperanza de vida que pasar de 50.000 a 58.000. En escala logarítmica se vuelve casi una recta (es la llamada *curva de Preston*).

In [2]:
base = alt.Chart(corte).mark_circle(opacity=0.6, color="#2a78d6").encode(
    y=alt.Y("vida:Q", scale=alt.Scale(domain=[50, 88]), title="Esperanza de vida (años)"),
    tooltip=["pais", "pib", "vida"],
).properties(width=330, height=260)

base.encode(x=alt.X("pib:Q", title="PIB per cápita PPA (lineal)")) | \
base.encode(x=alt.X("pib:Q", scale=alt.Scale(type="log"), title="PIB per cápita PPA (log)"))


alt.HConcatChart(...)

Ajustamos una recta por mínimos cuadrados: `vida = a + b · ln(PIB)`.

In [3]:
x, y = np.log(corte["pib"]), corte["vida"]
b, a = np.polyfit(x, y, 1)
r2 = np.corrcoef(x, y)[0, 1] ** 2
print(f"vida = {a:.2f} + {b:.2f} · ln(PIB)    R² = {r2:.2f}")
print(f"Cada vez que el ingreso se duplica, la esperanza de vida esperada sube {b * np.log(2):.1f} años.")


vida = 22.85 + 5.26 · ln(PIB)    R² = 0.74
Cada vez que el ingreso se duplica, la esperanza de vida esperada sube 3.6 años.


El ingreso explica alrededor de **tres cuartas partes** de la diferencia en esperanza de vida entre países. La cuarta parte restante es la que nos interesa: países que viven bastante más (o bastante menos) de lo que su ingreso haría prever.

## 2. La brecha: años de más y años de menos

Definimos la **brecha** como la diferencia entre la esperanza de vida observada y la esperada según la recta.

In [4]:
corte["esperada"] = a + b * np.log(corte["pib"])
corte["brecha"] = corte["vida"] - corte["esperada"]

# Para el ranking dejamos afuera microestados (< 1 millón de habitantes): con poblaciones tan chicas
# las estimaciones son más inestables y desplazarían a países más relevantes para la nota.
grandes = corte[corte["poblacion"] >= 1_000_000]
cols = ["pais", "region", "pib", "vida", "esperada", "brecha"]
print("Viven MÁS de lo esperado")
display(grandes.nlargest(12, "brecha")[cols].round(1))
print("Viven MENOS de lo esperado")
display(grandes.nsmallest(12, "brecha")[cols].round(1))


Viven MÁS de lo esperado


,pais,region,pib,vida,esperada,brecha
3526,Jordania,"Medio Oriente, norte de África, Afganistán y P...",10468.0,78.0,71.5,6.5
3922,Líbano,"Medio Oriente, norte de África, Afganistán y P...",10469.0,77.9,71.5,6.4
4966,Malaui,África subsahariana,1634.0,67.6,61.7,5.8
5182,Nicaragua,América Latina y el Caribe,7661.0,75.1,69.9,5.2
3850,Kuwait,"Medio Oriente, norte de África, Afganistán y P...",46664.0,84.6,79.3,5.2
4102,Sri Lanka,Asia meridional,13793.0,77.7,72.9,4.7
4354,Marruecos,"Medio Oriente, norte de África, Afganistán y P...",9163.0,75.5,70.8,4.7
3562,Japón,Asia oriental y Pacífico,47480.0,84.0,79.4,4.6
7666,Kosovo,Europa y Asia central,15716.0,78.2,73.6,4.6
2050,Ecuador,América Latina y el Caribe,13922.0,77.6,73.0,4.6


Viven MENOS de lo esperado


,pais,region,pib,vida,esperada,brecha
5146,Nigeria,África subsahariana,7994.0,54.6,70.1,-15.4
2734,Guinea Ecuatorial,África subsahariana,14998.0,63.9,73.4,-9.5
6802,Chad,África subsahariana,2431.0,55.2,63.8,-8.6
6622,Esuatini,África subsahariana,10386.0,64.3,71.5,-7.2
1366,Côte d’Ivoire,África subsahariana,6747.0,62.1,69.2,-7.1
4138,Lesoto,África subsahariana,2698.0,57.8,64.4,-6.6
7738,Sudáfrica,África subsahariana,13598.0,66.3,72.9,-6.6
2446,Gabón,África subsahariana,18924.0,68.5,74.6,-6.1
106,Angola,África subsahariana,8753.0,64.8,70.6,-5.7
2626,Guinea,África subsahariana,4016.0,60.9,66.5,-5.6


In [5]:
alt.Chart(corte).mark_boxplot(color="#2a78d6").encode(
    x=alt.X("brecha:Q", title="Brecha (años)"),
    y=alt.Y("region:N", title=None, sort=alt.EncodingSortField("brecha", op="median", order="descending")),
).properties(width=480, title="Brecha por región")


alt.Chart(...)

Lo que muestran las tablas:

- **Por encima de la recta** aparecen casos muy distintos: Japón entre los ricos; Costa Rica, Chile y Ecuador en América Latina; Sri Lanka y Bangladés en Asia meridional; Jordania, Líbano y Marruecos en Medio Oriente y el norte de África.
- **Por debajo** domina África subsahariana. Ahí se mezclan países con alta prevalencia de VIH (Sudáfrica, Esuatini, Lesoto) y **economías petroleras** donde el PIB per cápita es alto pero no se traduce en salud (Nigeria, Guinea Ecuatorial, Gabón, Angola). Entre los microestados excluidos del ranking pasa lo mismo con Guyana y Brunéi.
- Entre los países de **ingreso alto**, los mayores déficits son Rusia, Trinidad y Tobago y **Estados Unidos**.

### ¿Depende de cómo armamos el modelo?

La recta es una simplificación. Probamos dos alternativas: una curva cuadrática en el logaritmo del ingreso y una regresión ponderada por población, para que los países grandes pesen más. Si las brechas cambiaran mucho, el ranking dependería de una decisión arbitraria nuestra.

In [6]:
lx = np.log(corte["pib"])
cuad = np.polyfit(lx, corte["vida"], 2)
bp, ap = np.polyfit(lx, corte["vida"], 1, w=np.sqrt(corte["poblacion"]))
alternativas = corte.assign(
    brecha_cuadratica=corte["vida"] - np.polyval(cuad, lx),
    brecha_ponderada=corte["vida"] - (ap + bp * lx),
)
print("Correlación con la brecha lineal:")
print(alternativas[["brecha", "brecha_cuadratica", "brecha_ponderada"]].corr().iloc[0].round(3))
alternativas[alternativas["iso3"].isin(["CRI", "CHL", "JPN", "ARG", "USA", "RUS", "NGA"])][
    ["pais", "brecha", "brecha_cuadratica", "brecha_ponderada"]].round(1)


Correlación con la brecha lineal:
brecha               1.000
brecha_cuadratica    0.998
brecha_ponderada     0.999
Name: brecha, dtype: float64


,pais,brecha,brecha_cuadratica,brecha_ponderada
250,Argentina,1.1,1.3,0.7
1294,Chile,4.3,4.4,3.9
1618,Costa Rica,4.5,4.6,4.1
3562,Japón,4.6,4.5,4.1
5146,Nigeria,-15.4,-15.2,-15.6
5974,Rusia,-5.3,-5.3,-5.8
7342,Estados Unidos,-3.0,-3.3,-3.6


Las tres versiones dan prácticamente el mismo resultado (correlaciones mayores a 0,99), así que nos quedamos con la recta simple, que es la más fácil de explicar en una nota.

## 3. ¿Es un fenómeno estable o un efecto de la pandemia?

2020–2021 golpearon la esperanza de vida de forma muy desigual. Comparamos la brecha de 2024 con la de 2019, calculada con su propia recta.

In [7]:
def brechas(anio):
    d = panel[panel["anio"] == anio].dropna(subset=["vida", "pib"])
    bb, aa = np.polyfit(np.log(d["pib"]), d["vida"], 1)
    return d.assign(brecha=d["vida"] - (aa + bb * np.log(d["pib"])))[["iso3", "pais", "brecha"]]

comp = brechas(2019).merge(brechas(ANIO), on=["iso3", "pais"], suffixes=("_2019", f"_{ANIO}"))
print(f"Correlación de la brecha 2019 vs {ANIO}: {comp['brecha_2019'].corr(comp[f'brecha_{ANIO}']):.2f}")
comp[comp["iso3"].isin(["CRI", "CHL", "USA", "ARG", "JPN", "RUS", "ZAF"])].round(1)


Correlación de la brecha 2019 vs 2024: 0.86


,iso3,pais,brecha_2019,brecha_2024
6,ARG,Argentina,0.8,1.1
33,CHL,Chile,4.0,4.3
42,CRI,Costa Rica,5.0,4.5
90,JPN,Japón,5.1,4.6
150,RUS,Rusia,-4.9,-5.3
185,USA,Estados Unidos,-2.9,-3.0
192,ZAF,Sudáfrica,-6.3,-6.6


La correlación es alta: los países que se salen del patrón lo hacían antes de la pandemia. No es un accidente de un año.

## 4. ¿Y el gasto en salud?

Una explicación intuitiva sería que los que viven más gastan más en salud. Tomamos el último dato de gasto per cápita disponible para cada país.

In [8]:
gasto = (panel.dropna(subset=["gasto_salud"]).sort_values("anio")
         .groupby("iso3").tail(1)[["iso3", "anio", "gasto_salud"]]
         .rename(columns={"anio": "anio_gasto"}))
corte = corte.drop(columns="gasto_salud").merge(gasto, on="iso3", how="left")

duckdb.sql("""
    SELECT pais, round(pib) AS pib, round(gasto_salud) AS gasto_pc, anio_gasto,
           round(vida, 1) AS vida, round(brecha, 1) AS brecha
    FROM corte
    WHERE iso3 IN ('USA', 'CRI', 'CHL', 'JPN', 'ESP', 'ARG', 'URY')
    ORDER BY gasto_pc DESC
""").df()


,pais,pib,gasto_pc,anio_gasto,vida,brecha
0,Estados Unidos,75698.0,13473.0,2023.0,78.9,-3.0
1,Japón,47480.0,5365.0,2023.0,84.0,4.6
2,España,48460.0,4935.0,2023.0,83.9,4.3
3,Chile,30268.0,3612.0,2024.0,81.4,4.3
4,Uruguay,32149.0,3135.0,2023.0,78.3,0.9
5,Argentina,26772.0,3089.0,2023.0,77.5,1.1
6,Costa Rica,27325.0,1928.0,2023.0,81.0,4.5


In [9]:
alt.Chart(corte.dropna(subset=["gasto_salud"])).mark_circle(opacity=0.6, color="#2a78d6").encode(
    x=alt.X("gasto_salud:Q", scale=alt.Scale(type="log"), title="Gasto en salud per cápita PPA (log)"),
    y=alt.Y("brecha:Q", title="Brecha (años)"),
    tooltip=["pais", "gasto_salud", "brecha"],
).properties(width=480, height=280)


alt.Chart(...)

In [10]:
g = corte.dropna(subset=["gasto_salud"])
print(f"Correlación entre la brecha y el gasto en salud (log): {np.corrcoef(np.log(g['gasto_salud']), g['brecha'])[0, 1]:.2f}")


Correlación entre la brecha y el gasto en salud (log): 0.09


In [11]:
usa, cri = corte.set_index("iso3").loc["USA"], corte.set_index("iso3").loc["CRI"]
print(f"Estados Unidos gasta {usa.gasto_salud / cri.gasto_salud:.1f} veces más por persona en salud que Costa Rica "
      f"y tiene {usa.pib / cri.pib:.1f} veces su ingreso, pero vive {cri.vida - usa.vida:.1f} años menos.")


Estados Unidos gasta 7.0 veces más por persona en salud que Costa Rica y tiene 2.8 veces su ingreso, pero vive 2.1 años menos.


El gasto no explica la brecha: una vez descontado el ingreso, gastar más casi no se asocia con vivir más de lo esperado (correlación ≈ 0,1). Estados Unidos es el caso extremo.



### Mortalidad de menores de 5 años

¿Cuánto cambia el ajuste al incorporar la mortalidad de menores de 5 años? Comparamos ambos modelos sobre las mismas economías de 2024 con datos completos.

In [12]:
m = corte.dropna(subset=["mortalidad_5"])
y_mort = m["vida"].to_numpy()
X = np.column_stack([np.ones(len(m)), np.log(m["pib"]), np.log(m["mortalidad_5"])])
X_pib = X[:, :2]
coef_pib, *_ = np.linalg.lstsq(X_pib, y_mort, rcond=None)
coef, *_ = np.linalg.lstsq(X, y_mort, rcond=None)

suma_cuadrados_total = np.sum((y_mort - y_mort.mean()) ** 2)
r2_pib_muestra_mort = 1 - np.sum((y_mort - X_pib @ coef_pib) ** 2) / suma_cuadrados_total
r2_mort = 1 - np.sum((y_mort - X @ coef) ** 2) / suma_cuadrados_total
print(f"Muestra común: {len(m)} economías")
print(f"R² solo con PIB: {r2_pib_muestra_mort:.4f} · con PIB y mortalidad de menores de 5 años: {r2_mort:.4f}")


Muestra común: 186 economías
R² solo con PIB: 0.7339 · con PIB y mortalidad de menores de 5 años: 0.8452


Entre las 186 economías con datos completos, el modelo basado en el PIB per cápita da cuenta del 73,4 % de la variación observada en la esperanza de vida. Al incorporar la mortalidad de menores de 5 años, la proporción sube al 84,5 %. Parte de esa mejora refleja que la esperanza de vida al nacer ya incorpora la mortalidad en esas edades. La comparación mide el ajuste dentro de la muestra y no permite estimar un efecto causal.

### ¿Mujeres o varones?

En casi todos los países las mujeres viven más. Calculamos la brecha por separado para cada sexo, con su propia recta.

In [13]:
for col in ["vida_mujeres", "vida_varones"]:
    d = corte.dropna(subset=[col])
    bb, aa = np.polyfit(np.log(d["pib"]), d[col], 1)
    corte["brecha_" + col.split("_")[1]] = corte[col] - (aa + bb * np.log(corte["pib"]))
corte["diferencia_sexos"] = corte["vida_mujeres"] - corte["vida_varones"]

print(f"Diferencia mediana entre mujeres y varones: {corte['diferencia_sexos'].median():.1f} años")
cols = ["pais", "vida_mujeres", "vida_varones", "diferencia_sexos", "brecha_mujeres", "brecha_varones"]
display(corte[corte["poblacion"] >= 1_000_000].nlargest(6, "diferencia_sexos")[cols].round(1))
corte[corte["iso3"].isin(["USA", "CRI", "ARG", "JPN", "NGA", "RUS"])][cols].round(1)


Diferencia mediana entre mujeres y varones: 5.0 años


,pais,vida_mujeres,vida_varones,diferencia_sexos,brecha_mujeres,brecha_varones
150,Rusia,78.9,68.3,10.6,-2.6,-7.9
108,Letonia,81.5,71.6,9.9,0.6,-4.0
183,Ucrania,79.4,69.8,9.6,2.9,-1.5
21,Bielorrusia,79.2,69.7,9.5,-0.3,-4.5
65,Georgia,79.2,69.8,9.5,0.7,-3.6
188,Vietnam,79.4,70.0,9.3,3.6,-0.6


,pais,vida_mujeres,vida_varones,diferencia_sexos,brecha_mujeres,brecha_varones
6,Argentina,80.0,75.0,5.0,0.9,1.1
42,Costa Rica,83.6,78.4,5.2,4.4,4.4
90,Japón,87.1,81.1,6.0,5.0,4.3
129,Nigeria,54.9,54.3,0.6,-17.7,-13.2
150,Rusia,78.9,68.3,10.6,-2.6,-7.9
185,Estados Unidos,81.4,76.5,4.9,-3.3,-2.7


El caso de Rusia es muy claro: las mujeres viven 2,6 años menos de lo esperado, pero **los varones, 7,9 años menos**. La diferencia entre sexos (10,6 años) es la mayor del mundo, y se repite en otros países de la ex Unión Soviética. En Estados Unidos, en cambio, el déficit es parejo entre mujeres y varones.

## 5. Trayectorias

¿Cómo llegaron hasta acá? Miramos el recorrido 1990–2024 de algunos países en el mismo plano ingreso–esperanza de vida.

In [14]:
tray = panel.dropna(subset=["vida", "pib"])
sel = tray[tray["iso3"].isin(["USA", "CRI", "CHL", "ARG"])]
alt.Chart(sel).mark_line(point=alt.OverlayMarkDef(size=12)).encode(
    x=alt.X("pib:Q", scale=alt.Scale(type="log"), title="PIB per cápita PPA (log)"),
    y=alt.Y("vida:Q", scale=alt.Scale(zero=False), title="Esperanza de vida (años)"),
    color=alt.Color("pais:N", title=None),
    order="anio:O",
    tooltip=["pais", "anio", "pib", "vida"],
).properties(width=480, height=300)


alt.Chart(...)

Desde 1990, Estados Unidos casi duplicó su ingreso por habitante, pero ganó menos de cuatro años de vida. Costa Rica y Chile partieron de un ingreso mucho menor y hoy lo superan en esperanza de vida.

## Exportación para la nota

In [15]:
salida = corte[["iso3", "pais", "region", "ingreso", "poblacion", "pib", "vida", "esperada", "brecha",
                "vida_mujeres", "vida_varones", "brecha_mujeres", "brecha_varones",
                "gasto_salud", "anio_gasto", "mortalidad_5"]]
salida.round({"pib": 0, "vida": 1, "esperada": 1, "brecha": 2, "vida_mujeres": 1, "vida_varones": 1,
              "brecha_mujeres": 1, "brecha_varones": 1, "gasto_salud": 0, "mortalidad_5": 1}) \
      .to_csv("../datos/paises_2024.csv", index=False)

tray[["iso3", "anio", "pib", "vida"]].round({"pib": 0, "vida": 1}).to_csv("../datos/trayectorias.csv", index=False)

pd.Series({"anio": ANIO, "a": round(a, 4), "b": round(b, 4), "r2": round(r2, 4), "r2_mortalidad": round(r2_mort, 4),
           "r2_pib_muestra_mortalidad": round(r2_pib_muestra_mort, 4), "paises_muestra_mortalidad": len(m),
           "paises": len(corte)}).to_json("../datos/modelo.json")
print(len(salida), "países ·", len(tray), "puntos de trayectoria")


195 países · 6792 puntos de trayectoria
